# 25K Museum Images with Captions — Starter EDA

This notebook loads the canonical Parquet table, checks rights/splits, plots category counts, and previews a small deterministic image grid. `model_text` is composed from Smithsonian metadata; it is not a generated visual caption.

In [ ]:
from pathlib import Path
import io
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

kaggle_roots = list(Path('/kaggle/input').glob('*/metadata.parquet'))
if kaggle_roots:
    DATA_ROOT = kaggle_roots[0].parent
else:
    DATA_ROOT = Path('../data/release/museum-images').resolve()

IMAGE_DIR = DATA_ROOT / 'images'
IMAGE_ZIP = DATA_ROOT / 'images.zip'
IMAGE_ARCHIVE = zipfile.ZipFile(IMAGE_ZIP) if (not IMAGE_DIR.is_dir() and IMAGE_ZIP.is_file()) else None

def load_release_image(file_name):
    if IMAGE_DIR.is_dir():
        return Image.open(IMAGE_DIR / file_name).convert('RGB')
    if IMAGE_ARCHIVE is None:
        raise FileNotFoundError('Neither images/ nor images.zip was found')
    return Image.open(io.BytesIO(IMAGE_ARCHIVE.read(file_name))).convert('RGB')

metadata_path = DATA_ROOT / 'metadata.parquet'
df = pd.read_parquet(metadata_path)
print('Data root:', DATA_ROOT)
print('Rows:', len(df), 'Objects:', df['object_id'].nunique())
df.head(3)

## Integrity checks

In [ ]:
assert df['image_id'].is_unique
assert df['file_name'].is_unique
assert (df['rights'] == 'CC0').all()
assert (df['metadata_rights'] == 'CC0').all()
assert (df['media_rights'] == 'CC0').all()
assert df['model_text'].fillna('').str.len().gt(0).mean() >= 0.90

object_split_counts = df.groupby('object_id')['split'].nunique()
assert object_split_counts.max() == 1
print('Integrity checks passed.')

## Category and split distribution

In [ ]:
category_counts = df['category'].value_counts().sort_values()
ax = category_counts.plot(kind='barh', figsize=(9, 5), title='Images by broad category')
ax.set_xlabel('rows')
plt.tight_layout()
plt.show()

pd.crosstab(df['category'], df['split'])

## Deterministic image-text preview

In [ ]:
sample = df.sample(n=min(12, len(df)), random_state=20260910).reset_index(drop=True)
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, row in zip(axes.flat, sample.to_dict('records')):
    image = load_release_image(row['file_name'])
    ax.imshow(image)
    title = str(row['model_text'])
    ax.set_title(title[:100] + ('…' if len(title) > 100 else ''), fontsize=8)
    ax.axis('off')
for ax in axes.flat[len(sample):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Starter 5K subset

In [ ]:
starter_path = DATA_ROOT / 'starter_5k' / 'metadata.parquet'
if starter_path.is_file():
    starter = pd.read_parquet(starter_path)
else:
    with zipfile.ZipFile(DATA_ROOT / 'starter_5k.zip') as archive:
        starter = pd.read_parquet(io.BytesIO(archive.read('metadata.parquet')))
print('Starter rows:', len(starter), 'Starter objects:', starter['object_id'].nunique())
starter['category'].value_counts().sort_index()

## Training path example

For most frameworks, use `file_name` to join each row to `images/<file_name>` and use `model_text` as the compact text input. Keep `object_id` available for any custom resampling or evaluation so multiple views remain grouped.